In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Cargar dataset
# ========================
df = pd.read_csv('data/raw/diabetic_data.csv')


# 2. Preparacion de con funcione central
# ========================

# Diccionario para almacenar los resultados de evaluación
evaluation_results = []
metrics_df = pd.DataFrame() # DataFrame final para la tabla de métricas



def evaluate_model(model_name, model_instance, X_test, y_test, is_scaled=False):
    """
    Entrena el modelo (si no está entrenado) y realiza la evaluación de métricas
    y visualizaciones.
    """
    print(f"\n===== EVALUANDO: {model_name} =====")
    
    # Usar datos escalados si el modelo lo requiere
    X_to_use = X_test if not is_scaled else X_test_scaled
    
    
    if hasattr(model_instance, 'predict_proba'):
        y_pred_proba = model_instance.predict_proba(X_to_use)[:, 1]
    elif hasattr(model_instance, 'decision_function'):
        y_scores = model_instance.decision_function(X_to_use)
        # Normalizar a probabilidades para ROC-AUC
        y_pred_proba = y_scores 
    else:
        y_pred_proba = model_instance.predict(X_to_use) 

    # Predicciones binarias
    y_pred = model_instance.predict(X_to_use)
    
    # Calcular Métricas
    try:
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        # Solo calcular AUC si hay scores/probabilidades razonables
        auc = roc_auc_score(y_test, y_pred_proba)
    except Exception as e:
        # Esto puede ocurrir con Lasso o Ridge si se usa la implementación de sklearn
        # que no es de clasificación, o si hay un error en predict_proba.
        print(f"Error calculando métricas para {model_name}: {e}")
        accuracy, precision, recall, f1, auc = np.nan, np.nan, np.nan, np.nan, np.nan
        
    
    metrics = {
        'Modelo': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'AUC': auc,
    }
    evaluation_results.append(metrics)
    
    # 2.2 Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Clase 0', 'Clase 1'], yticklabels=['Clase 0', 'Clase 1'])
    plt.title(f'Matriz de Confusión: {model_name}')
    plt.ylabel('Valores Reales')
    plt.xlabel('Predicciones')
    plt.show()
    
    # 2.3 Curva ROC
    if not np.isnan(auc):
        fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
        plt.figure(figsize=(5, 4))
        plt.plot(fpr, tpr, label=f'ROC {model_name} (AUC = {auc:.4f})')
        plt.plot([0, 1], [0, 1], 'r--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('Tasa de Falsos Positivos (FPR)')
        plt.ylabel('Tasa de Verdaderos Positivos (TPR) - Recall')
        plt.title('Curva ROC')
        plt.legend(loc="lower right")
        plt.show()
    
    return model_instance

In [ ]:
#  KNN
knn_model = KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree', n_jobs=-1)
knn_model.fit(X_train_scaled, y_train)
evaluate_model("KNN", knn_model, X_test, y_test, is_scaled=True)

# Naive Bayes
gnb_model = GaussianNB()
gnb_model.fit(X_train_scaled, y_train) # Naive Bayes funciona bien con datos escalados
evaluate_model("Naive Bayes", gnb_model, X_test, y_test, is_scaled=True)

#  Regresión Logística (L1 y L2)
lr_l1_model = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, n_jobs=-1)
lr_l1_model.fit(X_train_scaled, y_train)
evaluate_model("LogReg (L1)", lr_l1_model, X_test, y_test, is_scaled=True)

lr_l2_model = LogisticRegression(penalty='l2', solver='lbfgs', random_state=42, n_jobs=-1)
lr_l2_model.fit(X_train_scaled, y_train)
evaluate_model("LogReg (L2)", lr_l2_model, X_test, y_test, is_scaled=True)


#comparar con los resultados de optimización a ver si se usa saga o no
# Ridge
# Usamos RidgeClassifier que es para clasificación y tiene función de decisión
ridge_model = RidgeClassifier(alpha=1.0, solver='saga', random_state=42)
ridge_model.fit(X_train_scaled, y_train)
evaluate_model("RidgeClassifier", ridge_model, X_test, y_test, is_scaled=True)

#  Lasso (Implementado como LogReg L1 con solver saga)
lasso_model = LogisticRegression(penalty='l1', solver='saga', random_state=42, n_jobs=-1)
lasso_model.fit(X_train_scaled, y_train)
evaluate_model("Lasso (LogReg L1-Saga)", lasso_model, X_test, y_test, is_scaled=True)

#  Árbol de Decisión
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train) # Árboles no requieren escalado
evaluate_model("Árbol de Decisión", dt_model, X_test, y_test, is_scaled=False)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train) # Árboles no requieren escalado
evaluate_model("Random Forest", rf_model, X_test, y_test, is_scaled=False)

#  XGBoost
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    tree_method='hist',
    n_estimators=100, # Bajamos el número para el reentrenamiento simple
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train, y_train) # XGBoost no requiere escalado
evaluate_model("XGBoost", xgb_model, X_test, y_test, is_scaled=False)

#  SVM Lineal
# Usamos LinearSVC, que es rápido y soporta SGD.
svm_linear_model = LinearSVC(random_state=42, dual=False)
svm_linear_model.fit(X_train_scaled, y_train)
evaluate_model("SVM Lineal (LinearSVC)", svm_linear_model, X_test, y_test, is_scaled=True)

#  RESULTADO FINAL

# 4Generar Tabla de Métricas (Punto 5 del documento)
metrics_df = pd.DataFrame(evaluation_results)
metrics_df = metrics_df.set_index('Modelo').sort_values(by='AUC', ascending=False)

print("\n\n#####################################################")
print("### TABLA DE MÉTRICAS FINAL (CLASIFICACIÓN) ###")
print("#####################################################")
print(metrics_df.to_markdown())